# Занятие 3, пара 2. Шум, нелинейные фильтры и масштаб


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


Картинка стала гладкой. Это хорошо? Смотря что мы хотели сохранить. Сегодня добавим два разных шума, попробуем их убрать и выясним, где красивый результат начинает терять полезные детали.

Работаем с CPU и уже знакомыми библиотеками. Нужны понятия массива, фильтра и среднего. Получим сравнение, которое можно повторить, и объясним хотя бы один компромисс.


### Подготовка

Работайте в своей копии внутри папки проекта. Запустите ячейки по порядку. Эта тетрадь не использует переменные из другой тетради. GitHub — по желанию; сохраняйте файл и отдельную копию.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


In [ ]:
import csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
from course_support.cvtools import imread, imwrite, show
OUTPUT_DIR = WORKSPACE_DIR / 'reports' / 'filters'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean = imread(DATA_DIR / 'hopper.jpg', cv2.IMREAD_GRAYSCALE)
clean = cv2.resize(clean, (256, 300), interpolation=cv2.INTER_AREA)
show(clean, titles=['Эталон для нашего эксперимента'])


## Два шума, два характера
Гауссов шум понемногу меняет много пикселей. «Соль и перец» заменяет часть пикселей на белые и чёрные. Это учебные модели: настоящая камера может давать более сложные помехи.

Сначала переводим значения в float, прибавляем шум, затем ограничиваем диапазон и возвращаем uint8. Иначе арифметика uint8 может переполниться. `seed` фиксирует случайный опыт: сравним фильтры на **одной и той же** испорченной картинке.


In [ ]:
def add_gaussian(image, sigma=20, seed=42):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, sigma, image.shape)
    return np.clip(image.astype(float) + noise, 0, 255).astype(np.uint8)

def add_impulse(image, amount=0.08, seed=42):
    if not 0 <= amount <= 1:
        raise ValueError('Доля шума должна быть от 0 до 1')
    rng = np.random.default_rng(seed)
    mask = rng.random(image.shape)
    noisy = image.copy()
    noisy[mask < amount / 2] = 0
    noisy[mask > 1 - amount / 2] = 255
    return noisy

gaussian_noisy = add_gaussian(clean)
impulse_noisy = add_impulse(clean)
show(clean, gaussian_noisy, impulse_noisy, titles=['Эталон', 'Гауссов шум', 'Соль и перец'])


### ✏️ Ваш ход 1: одна управляемая помеха

С помощью готовой функции создайте `my_noisy`: «соль и перец», amount=0.08, seed=42. Сравните с оригиналом. Затем отдельно повторите с amount=0.2: изменился ли характер шума или только его количество?
Проверку запускайте для первого варианта. Предположите, поможет ли обычное среднее 3×3 убрать яркие точки.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.2.1', my_noisy, clean)


## Почему медиана помогает с «солью»?
В окошке восемь десяток и одна яркая точка 255. Среднее подтянется вверх, а медиана после сортировки останется 10. Один очень яркий сосед уже не командует всем двором.

Но и медиана не волшебная: мелкие линии могут исчезнуть. Сравним одинаковый размер окна, а потом увеличим только его.


In [ ]:
values = np.array([10, 10, 10, 10, 255, 10, 10, 10, 10])
print('Среднее:', values.mean(), 'медиана:', np.median(values))
averaged = cv2.blur(impulse_noisy, (3, 3))
show(impulse_noisy, averaged, titles=['Соль и перец', 'Среднее 3×3'])


### ✏️ Ваш ход 2: попробуйте медиану

Получите `median_result = cv2.medianBlur(my_noisy, 3)`, покажите рядом с my_noisy и clean. Затем в отдельном опыте попробуйте окно 7. Что стало с шумом и мелкими деталями? Одной фразы «стало лучше» мало: укажите участок и изменение.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.2.2', median_result, my_noisy)


## Как сравнить численно?
Раз у нас есть оригинал, посчитаем средний квадрат ошибки (MSE) и PSNR. Для одинакового диапазона 0…255 и одного эталона больший PSNR означает меньшую MSE. Совпадение пиксель в пиксель даёт бесконечный PSNR.

Это одна мера, а не оценка «красоты» или полезности для любой задачи. Чуть смещённое, но хорошо читаемое изображение может получить плохой балл. На реальном снимке без эталона эту формулу напрямую применить не к чему.

$$\operatorname{PSNR}=10\log_{10}\frac{255^2}{\operatorname{MSE}}.$$

Функция уже готова; сегодня важнее поставить честный эксперимент, чем воспроизвести формулу по памяти.


In [ ]:
def psnr(reference, result):
    if reference.shape != result.shape:
        raise ValueError('Нужны изображения одинакового размера')
    mse = np.mean((reference.astype(np.float64) - result.astype(np.float64)) ** 2)
    return float('inf') if mse == 0 else float(10 * np.log10(255**2 / mse))

def candidates(noisy):
    return {
        'Без фильтра': noisy,
        'Среднее 3': cv2.blur(noisy, (3, 3)),
        'Гаусс 3': cv2.GaussianBlur(noisy, (3, 3), sigmaX=1),
        'Медиана 3': cv2.medianBlur(noisy, 3),
        'Билатеральный': cv2.bilateralFilter(noisy, 5, 40, 3),
    }

scores = []
for noise_name, noisy in [('Гауссов', gaussian_noisy), ('Импульсный', impulse_noisy)]:
    variants = candidates(noisy)
    show(*variants.values(), titles=list(variants), cols=3, size=3)
    for method, result in variants.items():
        score = psnr(clean, result)
        scores.append({'noise': noise_name, 'method': method, 'psnr_db': round(score, 3)})
        print(f'{noise_name:12} | {method:16} | {score:.2f} dB')


### Наш вывод
Выберите один тип шума и один фильтр. Напишите: параметр, PSNR до/после, что видно глазами и что потерялось. По этой одной картинке нельзя объявить метод лучшим для всех фотографий.

Таблица ниже сохраняется в CSV. Картинка denoised.png — выбранная в упражнении медиана, а не автоматически «победитель всех сравнений».


In [ ]:
with (OUTPUT_DIR / 'comparison.csv').open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['noise', 'method', 'psnr_db'])
    writer.writeheader()
    writer.writerows(scores)
imwrite(OUTPUT_DIR / 'denoised.png', median_result)
print('Сохранено в', OUTPUT_DIR)


## «Увеличить!» — легко. Вернуть потерянное — сложнее
Уменьшим изображение в 4 раза, затем увеличим обратно. При уменьшении INTER_AREA учитывает области исходных пикселей; для увеличения сравним ближайшего соседа и линейную интерполяцию.

Ближайший сосед даёт заметные квадраты. Линейная интерполяция выглядит плавнее, но новые мелкие факты в сцене из неё не появляются. Помните порядок: `shape` — высота, ширина; размер в resize — ширина, высота.


### ✏️ Ваш ход 3: один размер, разные пиксели

Создайте `small_image` с шириной и высотой в 4 раза меньше clean, используя INTER_AREA. Верните исходный размер двумя способами: `up_nearest` через INTER_NEAREST, `up_linear` через INTER_LINEAR. Покажите оба варианта рядом с clean, сравните PSNR и выберите один участок с потерей деталей.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('03.2.3', small_image, up_nearest, up_linear, clean)


## Практический запас — 15–20 минут
1. Поменяйте seed и повторите сравнение на **одной новой** noisy. Победитель сохранился? А разница PSNR?
2. Выберите один crop с мелкими деталями и один с гладким фоном. Уместен ли один и тот же фильтр?
3. Сравните окно медианы 3 и 7 на импульсном шуме amount=0.2. Запишите, что улучшилось и чем заплатили.
4. Дополнительно: увеличьте small_image через INTER_CUBIC. Новая плавность ещё не доказывает, что мелкие детали восстановились.

Не обязательно делать всё. Один законченный опыт с объяснением полезнее четырёх картинок без вывода.

## Итог
Сохраните тетрадь, comparison.csv и denoised.png; проверьте отдельную копию. Объясните соседу, почему сравниваем на одном шуме и откуда у нас эталон. Это тренировка для части ЛР2 о фильтрации; полное задание после краёв и контуров. Баллы за активность и обязательная публикация на GitHub не добавляются.

По желанию: [OpenCV — фильтрация](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html), [книга Сзелиски](https://szeliski.org/Book/).


In [ ]:
summary('03.2')
